In [1]:
from sentence_transformers import SentenceTransformer
from elasticsearch import Elasticsearch

In [2]:
ELASTICSEARCH_URL = "http://localhost:9200"
INDEX_NAME = "ev-vehicles"

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

In [3]:
es_client = Elasticsearch(ELASTICSEARCH_URL)

print("Elasticsearch connected:", es_client.ping())

Elasticsearch connected: True


In [4]:
document_count = es_client.count(index=INDEX_NAME)["count"]

print(f"BEV documents in Elasticsearch: {document_count:,}")

NotFoundError: NotFoundError(404, 'index_not_found_exception', 'no such index [ev-vehicles]', ev-vehicles, index_or_alias)

In [6]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

print(f"Loaded embedding model: {EMBEDDING_MODEL_NAME}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loaded embedding model: sentence-transformers/all-MiniLM-L6-v2


In [7]:
SOURCE_FIELDS = [
    "id",
    "year",
    "make",
    "model",
    "vehicle_name",
    "vehicle_class",
    "drive",
    "electric_range_miles",
    "city_mpge",
    "highway_mpge",
    "combined_mpge",
    "charge_120v_hours",
    "charge_240v_hours",
    "ev_motor",
    "annual_fuel_cost_usd",
    "document_text",
]

In [8]:
def show_results(results):
    """Print retrieved BEV records in a readable format."""

    for rank, result in enumerate(results, start=1):
        source = result["_source"]

        print(f"Rank {rank}")
        print(f"Vehicle: {source.get('vehicle_name')}")
        print(f"FuelEconomy.gov ID: {source.get('id')}")
        print(f"Electric range: {source.get('electric_range_miles')} miles")
        print(f"Combined efficiency: {source.get('combined_mpge')} MPGe")
        print(f"Vehicle class: {source.get('vehicle_class')}")
        print(f"Score: {result.get('_score', 'N/A')}")
        print("-" * 80)

In [9]:
def text_search(query, number_of_results=5):
    """
    Search for BEVs using keyword and full-text matching.
    """

    response = es_client.search(
        index=INDEX_NAME,
        size=number_of_results,
        source=SOURCE_FIELDS,
        query={
            "multi_match": {
                "query": query,
                "fields": [
                    "vehicle_name^4",
                    "make^3",
                    "model^3",
                    "vehicle_class^2",
                    "document_text",
                ],
                "fuzziness": "AUTO",
            }
        },
    )

    return response["hits"]["hits"]

In [10]:
query = "Tesla Model 3"

text_results = text_search(query)

show_results(text_results)

Rank 1
Vehicle: 2022 Tesla Model 3 RWD
FuelEconomy.gov ID: 45013
Electric range: 272 miles
Combined efficiency: 132 MPGe
Vehicle class: Midsize Cars
Score: 34.119434
--------------------------------------------------------------------------------
Rank 2
Vehicle: 2023 Tesla Model 3 RWD
FuelEconomy.gov ID: 46206
Electric range: 272 miles
Combined efficiency: 132 MPGe
Vehicle class: Midsize Cars
Score: 34.119434
--------------------------------------------------------------------------------
Rank 3
Vehicle: 2024 Tesla Model 3 RWD
FuelEconomy.gov ID: 47909
Electric range: 272 miles
Combined efficiency: 132 MPGe
Vehicle class: Midsize Cars
Score: 34.119434
--------------------------------------------------------------------------------
Rank 4
Vehicle: 2026 Tesla Model 3 Performance
FuelEconomy.gov ID: 50036
Electric range: 309 miles
Combined efficiency: 114 MPGe
Vehicle class: Midsize Cars
Score: 34.119434
--------------------------------------------------------------------------------
Rank

In [11]:
query = "electric SUV with long range"

text_results = text_search(query)

show_results(text_results)

Rank 1
Vehicle: 2024 Hyundai Kona Electric Long Range
FuelEconomy.gov ID: 47449
Electric range: 261 miles
Combined efficiency: 116 MPGe
Vehicle class: Small Sport Utility Vehicle 2WD
Score: 33.769302
--------------------------------------------------------------------------------
Rank 2
Vehicle: 2024 Hyundai Kona Electric Standard Range
FuelEconomy.gov ID: 47831
Electric range: 200 miles
Combined efficiency: 118 MPGe
Vehicle class: Small Sport Utility Vehicle 2WD
Score: 22.522125
--------------------------------------------------------------------------------
Rank 3
Vehicle: 2025 Hyundai Kona Electric Standard Range
FuelEconomy.gov ID: 48359
Electric range: 200 miles
Combined efficiency: 118 MPGe
Vehicle class: Small Sport Utility Vehicle 2WD
Score: 22.522125
--------------------------------------------------------------------------------
Rank 4
Vehicle: 2025 Tesla Cybertruck Long Range
FuelEconomy.gov ID: 49152
Electric range: 335 miles
Combined efficiency: 82 MPGe
Vehicle class: Stan

In [12]:
def vector_search(query, number_of_results=5):
    """
    Search for BEVs using sentence-transformer vector similarity.
    """

    query_embedding = embedding_model.encode(
        query,
        normalize_embeddings=True,
    ).tolist()

    response = es_client.search(
        index=INDEX_NAME,
        size=number_of_results,
        source=SOURCE_FIELDS,
        knn={
            "field": "embedding",
            "query_vector": query_embedding,
            "k": number_of_results,
            "num_candidates": 100,
        },
    )

    return response["hits"]["hits"]

In [13]:
query = "Which fully electric cars can travel the farthest before recharging?"

vector_results = vector_search(query)

show_results(vector_results)

Rank 1
Vehicle: 2024 Dodge Charger 2-Dr Daytona R/T AWD 20in Goodyear
FuelEconomy.gov ID: 48783
Electric range: 268 miles
Combined efficiency: 85 MPGe
Vehicle class: Large Cars
Score: 0.77642226
--------------------------------------------------------------------------------
Rank 2
Vehicle: 2024 Dodge Charger 2-Dr Daytona R/T AWD 20in Nexen
FuelEconomy.gov ID: 48784
Electric range: 308 miles
Combined efficiency: 98 MPGe
Vehicle class: Large Cars
Score: 0.77581716
--------------------------------------------------------------------------------
Rank 3
Vehicle: 2025 Mercedes-Benz EQE 500 4matic (SUV)
FuelEconomy.gov ID: 48391
Electric range: 264 miles
Combined efficiency: 81 MPGe
Vehicle class: Midsize Station Wagons
Score: 0.7733977
--------------------------------------------------------------------------------
Rank 4
Vehicle: 2024 Dodge Charger 2-Dr Daytona R/T AWD 18in
FuelEconomy.gov ID: 48782
Electric range: 274 miles
Combined efficiency: 87 MPGe
Vehicle class: Large Cars
Score: 0.7

In [14]:
query = "Show efficient electric cars that use little energy"

vector_results = vector_search(query)

show_results(vector_results)

Rank 1
Vehicle: 2026 Porsche Macan 4S Electric
FuelEconomy.gov ID: 50295
Electric range: 290 miles
Combined efficiency: 92 MPGe
Vehicle class: Small Sport Utility Vehicle 4WD
Score: 0.77103305
--------------------------------------------------------------------------------
Rank 2
Vehicle: 2026 Lexus RZ 450e AWD (20 inch wheels - 235/50R20,255/45R20)
FuelEconomy.gov ID: 50219
Electric range: 228 miles
Combined efficiency: 95 MPGe
Vehicle class: Small Sport Utility Vehicle 4WD
Score: 0.769799
--------------------------------------------------------------------------------
Rank 3
Vehicle: 2027 Mercedes-Benz EQS 400 4matic (SUV)
FuelEconomy.gov ID: 50662
Electric range: 312 miles
Combined efficiency: 79 MPGe
Vehicle class: Standard Sport Utility Vehicle 4WD
Score: 0.76859665
--------------------------------------------------------------------------------
Rank 4
Vehicle: 2013 Fiat 500e
FuelEconomy.gov ID: 33396
Electric range: 87 miles
Combined efficiency: 116 MPGe
Vehicle class: Minicompac

In [15]:
query = "Which electric cars can drive the farthest without charging?"

In [16]:
print("TEXT SEARCH RESULTS")
print("=" * 80)

text_results = text_search(query)
show_results(text_results)

TEXT SEARCH RESULTS
Rank 1
Vehicle: 2011 smart fortwo electric drive cabriolet
FuelEconomy.gov ID: 31064
Electric range: 63 miles
Combined efficiency: 87 MPGe
Vehicle class: Two Seaters
Score: 30.487679
--------------------------------------------------------------------------------
Rank 2
Vehicle: 2011 smart fortwo electric drive coupe
FuelEconomy.gov ID: 31065
Electric range: 63 miles
Combined efficiency: 87 MPGe
Vehicle class: Two Seaters
Score: 30.487679
--------------------------------------------------------------------------------
Rank 3
Vehicle: 2013 smart fortwo electric drive convertible
FuelEconomy.gov ID: 33305
Electric range: 68 miles
Combined efficiency: 107 MPGe
Vehicle class: Two Seaters
Score: 30.487679
--------------------------------------------------------------------------------
Rank 4
Vehicle: 2013 smart fortwo electric drive coupe
FuelEconomy.gov ID: 33306
Electric range: 68 miles
Combined efficiency: 107 MPGe
Vehicle class: Two Seaters
Score: 30.487679
---------

In [17]:
print("VECTOR SEARCH RESULTS")
print("=" * 80)

vector_results = vector_search(query)
show_results(vector_results)

VECTOR SEARCH RESULTS
Rank 1
Vehicle: 2025 Tesla Model 3 Long Range RWD-I (19in wheels)
FuelEconomy.gov ID: 49154
Electric range: 346 miles
Combined efficiency: 131 MPGe
Vehicle class: Midsize Cars
Score: 0.7784989
--------------------------------------------------------------------------------
Rank 2
Vehicle: 2026 Tesla Model 3 Premium RWD
FuelEconomy.gov ID: 50038
Electric range: 363 miles
Combined efficiency: 137 MPGe
Vehicle class: Midsize Cars
Score: 0.7773638
--------------------------------------------------------------------------------
Rank 3
Vehicle: 2024 Tesla Model 3 Long Range RWD
FuelEconomy.gov ID: 48795
Electric range: 363 miles
Combined efficiency: 137 MPGe
Vehicle class: Midsize Cars
Score: 0.77476215
--------------------------------------------------------------------------------
Rank 4
Vehicle: 2024 Dodge Charger 2-Dr Daytona R/T AWD 20in Nexen
FuelEconomy.gov ID: 48784
Electric range: 308 miles
Combined efficiency: 98 MPGe
Vehicle class: Large Cars
Score: 0.7746658

In [18]:
def hybrid_search(
    query,
    number_of_results=5,
    candidate_results=20,
    rrf_constant=60,
):
    """
    Combine text and vector results using Reciprocal Rank Fusion.
    """

    text_results = text_search(
        query,
        number_of_results=candidate_results,
    )

    vector_results = vector_search(
        query,
        number_of_results=candidate_results,
    )

    combined_results = {}

    # Add scores based on text-search ranking
    for rank, result in enumerate(text_results, start=1):
        document_id = result["_id"]

        if document_id not in combined_results:
            combined_results[document_id] = {
                "_source": result["_source"],
                "_score": 0,
                "text_rank": None,
                "vector_rank": None,
            }

        combined_results[document_id]["_score"] += 1 / (
            rrf_constant + rank
        )

        combined_results[document_id]["text_rank"] = rank

    # Add scores based on vector-search ranking
    for rank, result in enumerate(vector_results, start=1):
        document_id = result["_id"]

        if document_id not in combined_results:
            combined_results[document_id] = {
                "_source": result["_source"],
                "_score": 0,
                "text_rank": None,
                "vector_rank": None,
            }

        combined_results[document_id]["_score"] += 1 / (
            rrf_constant + rank
        )

        combined_results[document_id]["vector_rank"] = rank

    ranked_results = sorted(
        combined_results.values(),
        key=lambda result: result["_score"],
        reverse=True,
    )

    return ranked_results[:number_of_results]

In [19]:
query = "Which battery electric vehicles have the longest driving range?"

hybrid_results = hybrid_search(query)

show_results(hybrid_results)

Rank 1
Vehicle: 2024 Hyundai Kona Electric Long Range
FuelEconomy.gov ID: 47449
Electric range: 261 miles
Combined efficiency: 116 MPGe
Vehicle class: Small Sport Utility Vehicle 2WD
Score: 0.01639344262295082
--------------------------------------------------------------------------------
Rank 2
Vehicle: 2020 Tesla Model S Long Range
FuelEconomy.gov ID: 42282
Electric range: 373 miles
Combined efficiency: 111 MPGe
Vehicle class: Large Cars
Score: 0.01639344262295082
--------------------------------------------------------------------------------
Rank 3
Vehicle: 2024 Hyundai Kona Electric Standard Range
FuelEconomy.gov ID: 47831
Electric range: 200 miles
Combined efficiency: 118 MPGe
Vehicle class: Small Sport Utility Vehicle 2WD
Score: 0.016129032258064516
--------------------------------------------------------------------------------
Rank 4
Vehicle: 2019 Tesla Model S Long Range
FuelEconomy.gov ID: 41417
Electric range: 370 miles
Combined efficiency: 111 MPGe
Vehicle class: Large Ca

In [20]:
query = "What are the specifications of the Tesla Model Y?"

hybrid_results = hybrid_search(query)

show_results(hybrid_results)

Rank 1
Vehicle: 2022 Tesla Model Y RWD
FuelEconomy.gov ID: 45017
Electric range: 244 miles
Combined efficiency: 129 MPGe
Vehicle class: Small Sport Utility Vehicle 2WD
Score: 0.032266458495966696
--------------------------------------------------------------------------------
Rank 2
Vehicle: 2024 Tesla Model Y RWD
FuelEconomy.gov ID: 48476
Electric range: 260 miles
Combined efficiency: 120 MPGe
Vehicle class: Small Sport Utility Vehicle 2WD
Score: 0.031754032258064516
--------------------------------------------------------------------------------
Rank 3
Vehicle: 2020 Tesla Model Y Performance AWD
FuelEconomy.gov ID: 42474
Electric range: 315 miles
Combined efficiency: 121 MPGe
Vehicle class: Small Sport Utility Vehicle 4WD
Score: 0.029877369007803793
--------------------------------------------------------------------------------
Rank 4
Vehicle: 2023 Tesla Model Y Performance AWD
FuelEconomy.gov ID: 46213
Electric range: 303 miles
Combined efficiency: 111 MPGe
Vehicle class: Small Spo

In [21]:
query = "Which electric cars have high MPGe efficiency?"

hybrid_results = hybrid_search(query)

show_results(hybrid_results)

Rank 1
Vehicle: 1998 Chevrolet S10 Electric
FuelEconomy.gov ID: 30976
Electric range: 33 miles
Combined efficiency: 55 MPGe
Vehicle class: Small Pickup Trucks 2WD
Score: 0.01639344262295082
--------------------------------------------------------------------------------
Rank 2
Vehicle: 2026 Lucid Air Pure RWD with 19 inch wheels
FuelEconomy.gov ID: 49969
Electric range: 420 miles
Combined efficiency: 146 MPGe
Vehicle class: Large Cars
Score: 0.01639344262295082
--------------------------------------------------------------------------------
Rank 3
Vehicle: 1998 Chevrolet S10 Electric
FuelEconomy.gov ID: 30977
Electric range: 72 miles
Combined efficiency: 28 MPGe
Vehicle class: Small Pickup Trucks 2WD
Score: 0.016129032258064516
--------------------------------------------------------------------------------
Rank 4
Vehicle: 2027 Mercedes-Benz EQE 320 Plus (SUV)
FuelEconomy.gov ID: 50661
Electric range: 302 miles
Combined efficiency: 93 MPGe
Vehicle class: Midsize Station Wagons
Score: 0

In [22]:
query = "electric SUV with long range"

hybrid_results = hybrid_search(query)

for rank, result in enumerate(hybrid_results, start=1):
    source = result["_source"]

    print(f"Final hybrid rank: {rank}")
    print(f"Vehicle: {source['vehicle_name']}")
    print(f"RRF score: {result['_score']:.4f}")
    print(f"Text-search rank: {result['text_rank']}")
    print(f"Vector-search rank: {result['vector_rank']}")
    print("-" * 80)

Final hybrid rank: 1
Vehicle: 2024 Hyundai Kona Electric Long Range
RRF score: 0.0164
Text-search rank: 1
Vector-search rank: None
--------------------------------------------------------------------------------
Final hybrid rank: 2
Vehicle: 2024 Rivian R1T All-Terrain Quad Large (20in)
RRF score: 0.0164
Text-search rank: None
Vector-search rank: 1
--------------------------------------------------------------------------------
Final hybrid rank: 3
Vehicle: 2024 Hyundai Kona Electric Standard Range
RRF score: 0.0161
Text-search rank: 2
Vector-search rank: None
--------------------------------------------------------------------------------
Final hybrid rank: 4
Vehicle: 2027 Rivian R1T Premium Long Range (20in AT)
RRF score: 0.0161
Text-search rank: None
Vector-search rank: 2
--------------------------------------------------------------------------------
Final hybrid rank: 5
Vehicle: 2025 Hyundai Kona Electric Standard Range
RRF score: 0.0159
Text-search rank: 3
Vector-search rank: Non

In [23]:
test_questions = [
    "What is the range of the Tesla Model 3?",
    "Which battery electric vehicles have the longest driving range?",
    "Show electric SUVs with good range.",
    "Which fully electric cars are the most efficient?",
    "What EVs have the shortest Level 2 charging time?",
    "Show electric vehicles made by Nissan.",
    "Which electric cars have all-wheel drive?",
    "What electric cars are in the small SUV class?",
]

In [24]:
query = test_questions[2]

print("QUESTION:")
print(query)

print("\nTEXT SEARCH")
show_results(text_search(query))

print("\nVECTOR SEARCH")
show_results(vector_search(query))

print("\nHYBRID SEARCH")
show_results(hybrid_search(query))

QUESTION:
Show electric SUVs with good range.

TEXT SEARCH
Rank 1
Vehicle: 2024 Hyundai Kona Electric Long Range
FuelEconomy.gov ID: 47449
Electric range: 261 miles
Combined efficiency: 116 MPGe
Vehicle class: Small Sport Utility Vehicle 2WD
Score: 22.522125
--------------------------------------------------------------------------------
Rank 2
Vehicle: 2024 Hyundai Kona Electric Standard Range
FuelEconomy.gov ID: 47831
Electric range: 200 miles
Combined efficiency: 118 MPGe
Vehicle class: Small Sport Utility Vehicle 2WD
Score: 22.522125
--------------------------------------------------------------------------------
Rank 3
Vehicle: 2025 Hyundai Kona Electric Standard Range
FuelEconomy.gov ID: 48359
Electric range: 200 miles
Combined efficiency: 118 MPGe
Vehicle class: Small Sport Utility Vehicle 2WD
Score: 22.522125
--------------------------------------------------------------------------------
Rank 4
Vehicle: 1998 Chevrolet S10 Electric
FuelEconomy.gov ID: 30976
Electric range: 33 m

In [25]:
import os

from dotenv import load_dotenv
from openai import OpenAI

In [ ]:
load_dotenv()

True

In [27]:
openai_client = OpenAI()

In [ ]:
def build_context(search_results):
    """
    Turn retrieved Elasticsearch results into text
    that can be sent to the OpenAI model.
    """

    context_parts = []

    for rank, result in enumerate(search_results, start=1):
        document = result["_source"]

        context_part = f"""
            Source {rank}
            Vehicle name: {document.get("vehicle_name")}
            FuelEconomy.gov vehicle ID: {document.get("id")}

            {document.get("document_text")}
        """.strip()

        context_parts.append(context_part)

    return "\n\n".join(context_parts)

In [29]:
question = "Which battery electric vehicles have the longest driving range?"

results = hybrid_search(question)

context = build_context(results)

print(context)

Source 1
            Vehicle name: 2024 Hyundai Kona Electric Long Range
            FuelEconomy.gov vehicle ID: 47449

            Vehicle: 2024 Hyundai Kona Electric Long Range
FuelEconomy.gov vehicle ID: 47449
Vehicle type: Battery Electric Vehicle (BEV)
Vehicle class: Small Sport Utility Vehicle 2WD
Drive: Front-Wheel Drive
Transmission: Automatic (A1)
Electric range: 261 miles
City efficiency: 129 MPGe
Highway efficiency: 103 MPGe
Combined efficiency: 116 MPGe
240V charging time: 6.7 hours
Electric motor: 150 kW PMSM
Annual energy cost estimate: 650 USD
Tailpipe CO2 emissions: 0 grams per mile
Source: U.S. Department of Energy FuelEconomy.gov dataset.

Source 2
            Vehicle name: 2020 Tesla Model S Long Range
            FuelEconomy.gov vehicle ID: 42282

            Vehicle: 2020 Tesla Model S Long Range
FuelEconomy.gov vehicle ID: 42282
Vehicle type: Battery Electric Vehicle (BEV)
Vehicle class: Large Cars
Drive: All-Wheel Drive
Transmission: Automatic (A1)
Electric range

In [ ]:
def create_prompt(question, context):
    """
    Create the final user prompt for the OpenAI model.
    """

    prompt = f"""
      You are EV Assistant, a helpful assistant for battery electric vehicles.

      Use only the vehicle information in the provided FuelEconomy.gov context.

      Rules:
      - Answer only questions about battery electric vehicles (BEVs).
      - Do not make up facts, specifications, prices, incentives, availability,
        charging network information, or battery details that are not in the context.
      - If the answer is not available in the context, clearly say:
        "I do not have enough information in the FuelEconomy.gov records provided."
      - When giving a number, include its unit, such as miles, MPGe, or hours.
      - Mention the vehicle year, make, and model when relevant.
      - Cite the source number used for your answer, for example [Source 1].
      - Keep the answer clear and concise.

      User question:
      {question}

      FuelEconomy.gov context:
      {context}
    """.strip()

    return prompt

In [31]:
prompt = create_prompt(question, context)

print(prompt)

You are EV Assistant, a helpful assistant for battery electric vehicles.

Use only the vehicle information in the provided FuelEconomy.gov context.

Rules:
- Answer only questions about battery electric vehicles (BEVs).
- Do not make up facts, specifications, prices, incentives, availability,
  charging network information, or battery details that are not in the context.
- If the answer is not available in the context, clearly say:
  "I do not have enough information in the FuelEconomy.gov records provided."
- When giving a number, include its unit, such as miles, MPGe, or hours.
- Mention the vehicle year, make, and model when relevant.
- Cite the source number used for your answer, for example [Source 1].
- Keep the answer clear and concise.

User question:
Which battery electric vehicles have the longest driving range?

FuelEconomy.gov context:
Source 1
            Vehicle name: 2024 Hyundai Kona Electric Long Range
            FuelEconomy.gov vehicle ID: 47449

            Vehicle:

In [32]:
def generate_answer(question, context):
    """
    Send the question and retrieved context to OpenAI.
    """

    prompt = create_prompt(question, context)

    response = openai_client.chat.completions.create(
        model="gpt-5.4-mini",
        temperature=0,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a careful assistant. "
                    "Use only the provided vehicle records."
                ),
            },
            {
                "role": "user",
                "content": prompt,
            },
        ],
    )

    answer = response.choices[0].message.content

    return answer

In [33]:
def ask_ev_assistant(question, retrieval_method="hybrid"):
    """
    Answer a BEV question using retrieval plus OpenAI.

    retrieval_method can be:
    - "text"
    - "vector"
    - "hybrid"
    """

    if retrieval_method == "text":
        search_results = text_search(question)

    elif retrieval_method == "vector":
        search_results = vector_search(question)

    elif retrieval_method == "hybrid":
        search_results = hybrid_search(question)

    else:
        raise ValueError(
            "retrieval_method must be 'text', 'vector', or 'hybrid'."
        )

    context = build_context(search_results)

    answer = generate_answer(question, context)

    return answer, search_results

In [34]:
question = "Which battery electric vehicles have the longest driving range?"

answer, sources = ask_ev_assistant(
    question,
    retrieval_method="hybrid",
)

print(answer)

The BEV with the longest driving range in the provided FuelEconomy.gov records is the **2020 Tesla Model S Long Range**, with an **electric range of 373 miles**. [Source 2]

The next longest is the **2019 Tesla Model S Long Range**, with an **electric range of 370 miles**. [Source 4]


In [35]:
show_results(sources)

Rank 1
Vehicle: 2024 Hyundai Kona Electric Long Range
FuelEconomy.gov ID: 47449
Electric range: 261 miles
Combined efficiency: 116 MPGe
Vehicle class: Small Sport Utility Vehicle 2WD
Score: 0.01639344262295082
--------------------------------------------------------------------------------
Rank 2
Vehicle: 2020 Tesla Model S Long Range
FuelEconomy.gov ID: 42282
Electric range: 373 miles
Combined efficiency: 111 MPGe
Vehicle class: Large Cars
Score: 0.01639344262295082
--------------------------------------------------------------------------------
Rank 3
Vehicle: 2024 Hyundai Kona Electric Standard Range
FuelEconomy.gov ID: 47831
Electric range: 200 miles
Combined efficiency: 118 MPGe
Vehicle class: Small Sport Utility Vehicle 2WD
Score: 0.016129032258064516
--------------------------------------------------------------------------------
Rank 4
Vehicle: 2019 Tesla Model S Long Range
FuelEconomy.gov ID: 41417
Electric range: 370 miles
Combined efficiency: 111 MPGe
Vehicle class: Large Ca

In [36]:
question = "What is the electric range of the Tesla Model 3?"

answer, sources = ask_ev_assistant(question)

print(answer)
show_results(sources)

The Tesla Model 3 has different electric ranges depending on the year and trim in the FuelEconomy.gov records:

- 2017 Tesla Model 3 Long Range: 310 miles [Source 1]
- 2018 Tesla Model 3 Mid Range: 260 miles [Source 2]
- 2018 Tesla Model 3 Long Range: 310 miles [Source 5]
- 2019 Tesla Model 3 Standard Range: 220 miles [Source 3]
- 2020 Tesla Model 3 Standard Range: 220 miles [Source 4]

If you want, I can also help compare the ranges by year or trim.
Rank 1
Vehicle: 2017 Tesla Model 3 Long Range
FuelEconomy.gov ID: 39769
Electric range: 310 miles
Combined efficiency: 126 MPGe
Vehicle class: Midsize Cars
Score: 0.03131881575727918
--------------------------------------------------------------------------------
Rank 2
Vehicle: 2018 Tesla Model 3 Mid Range
FuelEconomy.gov ID: 41056
Electric range: 260 miles
Combined efficiency: 123 MPGe
Vehicle class: Midsize Cars
Score: 0.031024531024531024
--------------------------------------------------------------------------------
Rank 3
Vehicle: 2

In [37]:
question = "Which electric SUVs have the longest driving range?"

answer, sources = ask_ev_assistant(question)

print(answer)
show_results(sources)

In the provided FuelEconomy.gov records, the electric SUV with the longest driving range is the **2024 Hyundai Kona Electric Long Range** with **261 miles** of electric range [Source 1].

The other SUV listed is the **2024 Hyundai Kona Electric Standard Range**, with **200 miles** [Source 3], and the Rivian entries are **pickup trucks**, not SUVs [Source 2][Source 4].
Rank 1
Vehicle: 2024 Hyundai Kona Electric Long Range
FuelEconomy.gov ID: 47449
Electric range: 261 miles
Combined efficiency: 116 MPGe
Vehicle class: Small Sport Utility Vehicle 2WD
Score: 0.01639344262295082
--------------------------------------------------------------------------------
Rank 2
Vehicle: 2026 Rivian R1T All-Terrain Tri Max (20in)
FuelEconomy.gov ID: 49706
Electric range: 329 miles
Combined efficiency: 68 MPGe
Vehicle class: Standard Pickup Trucks 4WD
Score: 0.01639344262295082
--------------------------------------------------------------------------------
Rank 3
Vehicle: 2024 Hyundai Kona Electric Stand

In [38]:
question = "What electric cars have all-wheel drive?"

answer, sources = ask_ev_assistant(question)

print(answer)
show_results(sources)

The BEVs with all-wheel drive in the provided FuelEconomy.gov records are:

- 2024 Rolls-Royce Spectre Black Badge (23 inch wheels) [Source 2]
- 2026 Rolls-Royce Spectre Black Badge (22 inch Wheels) [Source 4]

I do not have enough information in the FuelEconomy.gov records provided to identify any other all-wheel-drive electric cars.
Rank 1
Vehicle: 2011 smart fortwo electric drive cabriolet
FuelEconomy.gov ID: 31064
Electric range: 63 miles
Combined efficiency: 87 MPGe
Vehicle class: Two Seaters
Score: 0.01639344262295082
--------------------------------------------------------------------------------
Rank 2
Vehicle: 2024 Rolls-Royce Spectre Black Badge (23 inch wheels)
FuelEconomy.gov ID: 47479
Electric range: 264 miles
Combined efficiency: 73 MPGe
Vehicle class: Compact Cars
Score: 0.01639344262295082
--------------------------------------------------------------------------------
Rank 3
Vehicle: 2011 smart fortwo electric drive coupe
FuelEconomy.gov ID: 31065
Electric range: 63 mi

In [39]:
question = "Which BEVs have short Level 2 charging times?"

answer, sources = ask_ev_assistant(question)

print(answer)
show_results(sources)

The BEVs with the shortest Level 2 charging times in the provided records are:

- **2027 Cadillac LYRIQ 19kW Charger** — **7 hours** [Source 4]
- **2021 Polestar 2** — **8 hours** [Source 1]

These are shorter than the other BEVs listed in the context.
Rank 1
Vehicle: 2021 Polestar 2
FuelEconomy.gov ID: 43294
Electric range: 233 miles
Combined efficiency: 92 MPGe
Vehicle class: Midsize Cars
Score: 0.01639344262295082
--------------------------------------------------------------------------------
Rank 2
Vehicle: 2027 Cadillac LYRIQ 11kW Charger
FuelEconomy.gov ID: 50612
Electric range: 326 miles
Combined efficiency: 92 MPGe
Vehicle class: Small Sport Utility Vehicle 2WD
Score: 0.01639344262295082
--------------------------------------------------------------------------------
Rank 3
Vehicle: 2024 Fisker Ocean Sport 20in
FuelEconomy.gov ID: 48470
Electric range: 231 miles
Combined efficiency: 92 MPGe
Vehicle class: Standard Sport Utility Vehicle 2WD
Score: 0.016129032258064516
---------

In [40]:
question = "Which electric car has the best autonomous driving system?"

answer, sources = ask_ev_assistant(question)

print(answer)

I do not have enough information in the FuelEconomy.gov records provided. The records list BEV range, efficiency, charging time, and motor details, but they do not include autonomous driving system information.


In [41]:
question = "Which electric SUVs have the longest driving range?"

In [42]:
text_answer, text_sources = ask_ev_assistant(
    question,
    retrieval_method="text",
)

print("TEXT RETRIEVAL ANSWER")
print("=" * 80)
print(text_answer)

print("\nTEXT RETRIEVAL SOURCES")
show_results(text_sources)

TEXT RETRIEVAL ANSWER
The electric SUVs in the provided records are the Hyundai Kona Electric models:

- **2024 Hyundai Kona Electric Long Range** — **261 miles** of electric range [Source 1]
- **2024 Hyundai Kona Electric Standard Range** — **200 miles** of electric range [Source 2]
- **2025 Hyundai Kona Electric Standard Range** — **200 miles** of electric range [Source 3]

So, the **2024 Hyundai Kona Electric Long Range** has the longest driving range among the electric SUVs listed here. [Source 1]

TEXT RETRIEVAL SOURCES
Rank 1
Vehicle: 2024 Hyundai Kona Electric Long Range
FuelEconomy.gov ID: 47449
Electric range: 261 miles
Combined efficiency: 116 MPGe
Vehicle class: Small Sport Utility Vehicle 2WD
Score: 22.522125
--------------------------------------------------------------------------------
Rank 2
Vehicle: 2024 Hyundai Kona Electric Standard Range
FuelEconomy.gov ID: 47831
Electric range: 200 miles
Combined efficiency: 118 MPGe
Vehicle class: Small Sport Utility Vehicle 2WD
S

In [43]:
vector_answer, vector_sources = ask_ev_assistant(
    question,
    retrieval_method="vector",
)

print("VECTOR RETRIEVAL ANSWER")
print("=" * 80)
print(vector_answer)

print("\nVECTOR RETRIEVAL SOURCES")
show_results(vector_sources)

VECTOR RETRIEVAL ANSWER
The electric SUV with the longest driving range in the provided records is the **2026 Rivian R1S All-Terrain Dual Max (20in)** with an **electric range of 370 miles** [Source 5].

For comparison, the **2026 Rivian R1S All-Terrain Tri Max (20in)** has **329 miles** of range [Source 4].

VECTOR RETRIEVAL SOURCES
Rank 1
Vehicle: 2026 Rivian R1T All-Terrain Tri Max (20in)
FuelEconomy.gov ID: 49706
Electric range: 329 miles
Combined efficiency: 68 MPGe
Vehicle class: Standard Pickup Trucks 4WD
Score: 0.79020333
--------------------------------------------------------------------------------
Rank 2
Vehicle: 2024 Rivian R1T All-Terrain Quad Large (20in)
FuelEconomy.gov ID: 47866
Electric range: 274 miles
Combined efficiency: 63 MPGe
Vehicle class: Standard Pickup Trucks 4WD
Score: 0.78739
--------------------------------------------------------------------------------
Rank 3
Vehicle: 2025 Rivian R1T All-Terrain Tri Max (20in)
FuelEconomy.gov ID: 48754
Electric range: 3

In [44]:
hybrid_answer, hybrid_sources = ask_ev_assistant(
    question,
    retrieval_method="hybrid",
)

print("HYBRID RETRIEVAL ANSWER")
print("=" * 80)
print(hybrid_answer)

print("\nHYBRID RETRIEVAL SOURCES")
show_results(hybrid_sources)

HYBRID RETRIEVAL ANSWER
The electric SUV in the provided FuelEconomy.gov records with the longest driving range is the **2024 Hyundai Kona Electric Long Range**, with **261 miles** of electric range. It is a **Small Sport Utility Vehicle 2WD** [Source 1].

The other SUV listed is the **2024 Hyundai Kona Electric Standard Range**, with **200 miles** of electric range [Source 3].

HYBRID RETRIEVAL SOURCES
Rank 1
Vehicle: 2024 Hyundai Kona Electric Long Range
FuelEconomy.gov ID: 47449
Electric range: 261 miles
Combined efficiency: 116 MPGe
Vehicle class: Small Sport Utility Vehicle 2WD
Score: 0.01639344262295082
--------------------------------------------------------------------------------
Rank 2
Vehicle: 2026 Rivian R1T All-Terrain Tri Max (20in)
FuelEconomy.gov ID: 49706
Electric range: 329 miles
Combined efficiency: 68 MPGe
Vehicle class: Standard Pickup Trucks 4WD
Score: 0.01639344262295082
--------------------------------------------------------------------------------
Rank 3
Vehic